In [54]:
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt',
    'names.txt'
)

('names.txt', <http.client.HTTPMessage at 0x1a34e798170>)

In [55]:
words = open('names.txt', 'r').read().splitlines()

In [56]:
import torch
N = torch.zeros((27, 27, 27))

In [57]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [58]:
b = {}
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for i in range(len(chs) - 2):
        trigram = chs[i], chs[i + 1], chs[i + 2]
        b[trigram] = b.get(trigram, 0) + 1

b

{('.', 'e', 'm'): 1,
 ('e', 'm', 'm'): 1,
 ('m', 'm', 'a'): 1,
 ('m', 'a', '.'): 1}

In [59]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for i in range(len(chs) - 2):
        ix1 = stoi[chs[i]]
        ix2 = stoi[chs[i + 1]]
        ix3 = stoi[chs[i + 2]]
        N[ix1, ix2, ix3] += 1

In [60]:
N.shape

torch.Size([27, 27, 27])

In [61]:
P = (N + 1).float()
P /= P.sum(2, keepdims = True)

In [62]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    out = []
    ix1, ix2 = 0, 0
    while True:
        p = P[ix1, ix2]
        ix3 = torch.multinomial(p, num_samples = 1, replacement = True, generator = g).item()
        out.append(itos[ix3])
        if ix3 == 0:
            break
        ix1 = ix2
        ix2 = ix3
    word = ''.join(out)
    print(word)

ce.
za.
zogh.
uriana.
kaydnevonimittain.


In [63]:
n = 0
log_likelihood = 0.0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for i in range(len(chs) - 2):
        ix1 = stoi[chs[i]]
        ix2 = stoi[chs[i + 1]]
        ix3 = stoi[chs[i + 2]]
        prob = P[ix1, ix2, ix3]
        log_prob = torch.log(prob)
        log_likelihood += log_prob
        n += 1

loss = -log_likelihood/n
loss

tensor(2.0927)

In [64]:
xs = []
ys = []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for i in range(len(chs) - 2):
        ix1 = stoi[chs[i]]
        ix2 = stoi[chs[i + 1]]
        ix3 = stoi[chs[i + 2]]
        xs.append(ix1 * 27 + ix2)
        ys.append(ix3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
xs.shape

torch.Size([196113])

In [65]:
import torch.nn.functional as F 
n = len(xs)
xenc = F.one_hot(xs, num_classes = 729).float()
xenc.shape

torch.Size([196113, 729])

In [66]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), dtype = torch.float32, requires_grad = True)

In [67]:
for i in range(200):
    logits = xenc @ W
    counts = logits.exp()
    probs = counts/counts.sum(1, keepdims = True)
    loss = (-probs[torch.arange(n), ys].log()).mean() + 0.01 * (W**2).mean()
    W.grad = None
    loss.backward()

    W.data += -85*W.grad

print(loss)

tensor(2.2029, grad_fn=<AddBackward0>)


In [68]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    ix = 0 # represents ..
    prev_char = stoi['.']
    out = []
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes = 729).float()
        logits = xenc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims = True)
        index = torch.multinomial(probs, num_samples = 1, replacement = True, generator = g).item()
        out.append(itos[index])
        if index == 0:
            break
        ix = prev_char * 27 + index
        prev_char = index
    word = ''.join(out)
    print(word)
    

dexzd.
zoglsuriana.
kayhkmdlqlmjttain.
luwak.
ka.
